## HyperSense — Dataset Extraction and Harmonization
### Objective
#### Prepare a harmonized adult hypertension dataset from:

- Ghana DHS 2014 Men (MR)
- Ghana DHS 2014 Women (IR)
- Benin DHS 2017–18 Men (MR)
- Benin DHS 2017–18 Women (IR); for development of a hypertension risk prediction model.

**Special Consideration:**
Benin DHS 2017–18 did not collect anthropometric measurements for men. Ghana DHS 2014 male BMI is available within the Household Recode (HR) dataset and must be merged into the Men's Recode (MR) dataset before extraction.

**Workflow:**
1. Verify Ghana male anthropometry variables.
2. Merge Ghana HR BMI into Ghana MR.
3. Extract required variables from Ghana Men.
4. Extract required variables from Ghana Women.
5. Extract required variables from Benin Men.
6. Extract required variables from Benin Women.
7. Standardize variable names and coding.
8. Generate harmonized datasets.
9. Save cleaned outputs for EDA and modeling.

**Key thing to note:** 
The datasets are being simultaenously explored with *IBM SPSS Statistics 25* for to increase merging and EDA speed

### Step 1 - Ghana Men BMI Recovery

In [81]:
import pandas as pd
import pyreadstat 

In [82]:
hr_df, meta = pyreadstat.read_sav(r"C:\Users\USER\Desktop\HyperSense\data\Ghana 2014\GHHR72SV\GHHR72FL.SAV")
print(hr_df.shape)

(11835, 3028)


In [83]:
print(hr_df.columns[:30].tolist())

['HHID', 'HV000', 'HV001', 'HV002', 'HV003', 'HV004', 'HV005', 'HV006', 'HV007', 'HV008', 'HV009', 'HV010', 'HV011', 'HV012', 'HV013', 'HV014', 'HV015', 'HV016', 'HV017', 'HV018', 'HV019', 'HV020', 'HV021', 'HV022', 'HV023', 'HV024', 'HV025', 'HV026', 'HV027', 'HV028']


In [84]:
# ------------------------------------------------------------------
# STEP 1A: Inspect Ghana HR anthropometry variables
# ------------------------------------------------------------------

In [85]:
# BMI
print([c for c in hr_df.columns if "HB40" in c])

['HB40$1', 'HB40$2', 'HB40$3', 'HB40$4', 'HB40$5', 'HB40$6', 'HB40$7', 'HB40$8']


In [86]:
# Age of household members
print([c for c in hr_df.columns if "HV105" in c])

['HV105$01', 'HV105$02', 'HV105$03', 'HV105$04', 'HV105$05', 'HV105$06', 'HV105$07', 'HV105$08', 'HV105$09', 'HV105$10', 'HV105$11', 'HV105$12', 'HV105$13', 'HV105$14', 'HV105$15', 'HV105$16', 'HV105$17', 'HV105$18', 'HV105$19', 'HV105$20', 'HV105$21', 'HV105$22', 'HV105$23', 'HV105$24', 'HV105$25']


In [87]:
# Sex of household member
print([c for c in hr_df.columns if "HV104" in c])

['HV104$01', 'HV104$02', 'HV104$03', 'HV104$04', 'HV104$05', 'HV104$06', 'HV104$07', 'HV104$08', 'HV104$09', 'HV104$10', 'HV104$11', 'HV104$12', 'HV104$13', 'HV104$14', 'HV104$15', 'HV104$16', 'HV104$17', 'HV104$18', 'HV104$19', 'HV104$20', 'HV104$21', 'HV104$22', 'HV104$23', 'HV104$24', 'HV104$25']


In [88]:
# Find all BMI-related columns in the Ghana Household Recode file
# Goal: determine how many household roster positions have BMI measurements

hb40_cols = [c for c in hr_df.columns if "HB40" in c]
print(sorted(hb40_cols))

['HB40$1', 'HB40$2', 'HB40$3', 'HB40$4', 'HB40$5', 'HB40$6', 'HB40$7', 'HB40$8']


In [89]:
# Find all HB41 (Rohrer's index) columns
# Goal: identify whether HB41 contains additional anthropometric measures

hb41_cols = [c for c in hr_df.columns if "HB41" in c]
print(sorted(hb41_cols))

['HB41$1', 'HB41$2', 'HB41$3', 'HB41$4', 'HB41$5', 'HB41$6', 'HB41$7', 'HB41$8']


In [90]:
# Confirm that HVIDX (Line number) exists for all roster positions

print(sorted([c for c in hr_df.columns if "HVIDX" in c]))

['HVIDX$01', 'HVIDX$02', 'HVIDX$03', 'HVIDX$04', 'HVIDX$05', 'HVIDX$06', 'HVIDX$07', 'HVIDX$08', 'HVIDX$09', 'HVIDX$10', 'HVIDX$11', 'HVIDX$12', 'HVIDX$13', 'HVIDX$14', 'HVIDX$15', 'HVIDX$16', 'HVIDX$17', 'HVIDX$18', 'HVIDX$19', 'HVIDX$20', 'HVIDX$21', 'HVIDX$22', 'HVIDX$23', 'HVIDX$24', 'HVIDX$25']


In [91]:
# Check how many BMI values exist in each roster slot

for i in range(1, 9):
    col = f"HB40${i}"
    print(col, hr_df[col].notna().sum())

HB40$1 3551
HB40$2 659
HB40$3 164
HB40$4 32
HB40$5 8
HB40$6 1
HB40$7 1
HB40$8 1


In [92]:
# ------------------------------------------------------------------
# STEP 1B: Identify merge keys between HR and MR datasets
# ------------------------------------------------------------------

### BMI Structure Verification - Ghana Household Recode

At this point, I have been able to confirm that the Ghana 2014 Household Recode (HR) dataset contains BMI measurements (`HB40`) and household roster line numbers (`HVIDX`).

However, before attempting any merge with the Men's Recode (MR) dataset, there's a need to understand exactly how DHS stored these BMI values.

The key question is:

> Does `HB40$1` belong to household member `HVIDX$1`, `HB40$2` belong to `HVIDX$2`, and so on?

If true, then we can safely reshape the household BMI data into a person-level table and merge it directly to the Men's Recode using:

- `HV001 ↔ MV001` (Cluster Number)
- `HV002 ↔ MV002` (Household Number)
- `HVIDX ↔ MV003` (Respondent Line Number)

Initial inspection already suggests this is the case:

- BMI values exist in `HB40$1`–`HB40$8`.
- Corresponding household member line numbers exist in `HVIDX$1`–`HVIDX$8`.
- BMI appears to be stored in DHS standard format (BMI × 100). Example: `1703 → 17.03 BMI`.

The inspection below is therefore a sanity check to verify the roster structure before reshaping and merging.

In [93]:
# First of all, inspect household containing BMI measurements

cols = [
    "HV001", "HV002",
    "HVIDX$01", "HV104$01", "HV105$01", "HB40$1",
    "HVIDX$02", "HV104$02", "HV105$02", "HB40$2"
]

hr_df.loc[hr_df["HB40$1"].notna(), cols].head()

,HV001,HV002,HVIDX$01,HV104$01,HV105$01,HB40$1,HVIDX$02,HV104$02,HV105$02,HB40$2
0,1.0,1.0,1.0,1.0,50.0,1703.0,2.0,2.0,42.0,NaN
1,1.0,2.0,1.0,1.0,72.0,1964.0,2.0,2.0,60.0,1909.0
2,1.0,3.0,1.0,1.0,27.0,2048.0,2.0,2.0,25.0,NaN
4,1.0,6.0,1.0,1.0,24.0,2108.0,2.0,2.0,19.0,NaN
9,1.0,11.0,1.0,1.0,40.0,1972.0,NaN,NaN,NaN,NaN


In [94]:
# Count households with members in roster positions 9–25
for i in range(9, 26):
    idx_col = f"HVIDX${i:02d}"

    if idx_col in hr_df.columns:
        print(idx_col, hr_df[idx_col].notna().sum())

HVIDX$09 499
HVIDX$10 286
HVIDX$11 170
HVIDX$12 125
HVIDX$13 77
HVIDX$14 57
HVIDX$15 36
HVIDX$16 23
HVIDX$17 18
HVIDX$18 11
HVIDX$19 8
HVIDX$20 4
HVIDX$21 3
HVIDX$22 1
HVIDX$23 1
HVIDX$24 1
HVIDX$25 1


In [95]:
# Confirm maximum roster position recorded
hvidx_cols = [c for c in hr_df.columns if "HVIDX" in c]
hr_df[hvidx_cols].max().max()

np.float64(25.0)

**Conclusion:** Household roster positions extend to 25 members, but BMI measurements (`HB40`) are only populated for slots 1–8, indicating that anthropometry was collected for a subset of eligible household members rather than the entire household roster.

In [96]:
# ------------------------------------------------------------------
# STEP 1C: Recover Ghana male BMI from household roster
# ------------------------------------------------------------------

##### Step 1C - Construct Ghana Male BMI Lookup Table

Objective:

Transform the household-level anthropometry variables (HB40$1-8) into a person-level BMI lookup table containing:

- HV001 (Cluster Number)
- HV002 (Household Number)
- HVIDX (Household Member Line Number)
- BMI

This lookup table will later be merged into the Ghana Men's Recode dataset using:

- HV001 ↔ MV001
- HV002 ↔ MV002
- HVIDX ↔ MV003

Expected output:

HV001 | HV002 | HVIDX | BMI

In [97]:
# Reshape household BMI roster from wide (HB40$1–HB40$8) to long format,
# keeping the corresponding household member index (HVIDX)

# Create empty list to store BMI records
bmi_records = []

# Loop through the 8 BMI slots in the household roster
for i in range(1, 9):

    bmi_col = f"HB40${i}"
    idx_col = f"HVIDX${i:02d}"

    # Keep only rows where BMI exists
    filtered_result = hr_df.loc[
        hr_df[bmi_col].notna(),
        ["HV001", "HV002", idx_col, bmi_col]
    ].copy()
    
    temp = filtered_result
    
    # Standardize column names
    temp.columns = ["HV001", "HV002", "HVIDX", "BMI"]

    bmi_records.append(temp)

# Combine all slots into one long table
bmi_lookup = pd.concat(bmi_records, ignore_index=True)

print(bmi_lookup.shape)
bmi_lookup.head()

(4417, 4)


,HV001,HV002,HVIDX,BMI
0,1.0,1.0,1.0,1703.0
1,1.0,2.0,1.0,1964.0
2,1.0,3.0,1.0,2048.0
3,1.0,6.0,1.0,2108.0
4,1.0,11.0,1.0,1972.0


In [98]:
# Last sanity check pre-merge: Check whether any person appears more than once

duplicates = bmi_lookup.duplicated(
    subset=["HV001", "HV002", "HVIDX"]
).sum()

print("Duplicate keys:", duplicates)

Duplicate keys: 0


In [99]:
# ------------------------------------------------------------------
# STEP 1D: Merge recovered BMI into Ghana Men's Recode
# ------------------------------------------------------------------

In [100]:
# Load Ghana Men Recode (MR)

gm_df, meta = pyreadstat.read_sav(
    r"C:\Users\USER\Desktop\HyperSense\data\Ghana 2014\GHMR71SV\GHMR71FL.SAV"
)

print(gm_df.shape)

(4388, 655)


In [101]:
# Verify merge identifiers exist

gm_df[["MV001", "MV002", "MV003"]].head()

,MV001,MV002,MV003
0,1.0,1.0,1.0
1,1.0,3.0,1.0
2,1.0,6.0,1.0
3,1.0,11.0,1.0
4,1.0,19.0,1.0


In [102]:
# Create merge-compatible keys

bmi_lookup = bmi_lookup.rename(
    columns={
        "HV001": "MV001",
        "HV002": "MV002",
        "HVIDX": "MV003"
    }
)

bmi_lookup.head()

,MV001,MV002,MV003,BMI
0,1.0,1.0,1.0,1703.0
1,1.0,2.0,1.0,1964.0
2,1.0,3.0,1.0,2048.0
3,1.0,6.0,1.0,2108.0
4,1.0,11.0,1.0,1972.0


In [103]:
# Merge recovered BMI into Ghana Men dataset

gm_df = gm_df.merge(
    bmi_lookup,
    on=["MV001", "MV002", "MV003"],
    how="left"
)

print(gm_df.shape)

(4388, 656)


In [104]:
# How many men received a BMI value?

print("Total men:", len(gm_df))
print("BMI available:", gm_df["BMI"].notna().sum())
print("BMI missing:", gm_df["BMI"].isna().sum())

Total men: 4388
BMI available: 3109
BMI missing: 1279


In [105]:
# Quick look at recovered BMI values

gm_df[["MV001", "MV002", "MV003", "BMI"]].head(10)

,MV001,MV002,MV003,BMI
0,1.0,1.0,1.0,1703.0
1,1.0,3.0,1.0,2048.0
2,1.0,6.0,1.0,2108.0
3,1.0,11.0,1.0,1972.0
4,1.0,19.0,1.0,2300.0
5,1.0,19.0,3.0,NaN
6,1.0,23.0,1.0,2794.0
7,1.0,26.0,1.0,1995.0
8,1.0,27.0,3.0,NaN
9,1.0,30.0,1.0,2074.0


### Step 2 — Extract Required Variables from Ghana Men (MR)

Now that BMI has been successfully recovered and merged into the Ghana Men's Recode dataset, the next step is to extract only the variables required for harmonization and modeling.

The goal is to reduce the full MR dataset to a Ghana Men analysis dataset containing demographic, behavioral, anthropometric, blood pressure, and survey design variables... thereafter doing the same for the three other datasets: Ghana Women, Benin Men, and Benin Women!

In [106]:
# --------------------------------------------------
# Variables required for Ghana Men dataset
# --------------------------------------------------

gm_vars = [
    "MCASEID", # Case Identification
    "MV001",   # cluster
    "MV002",   # household
    "MV003",   # respondent line number
    "MV005",   # Men's sample weight

    "MV012",   # age
    "MV106",   # education
    "MV025",   # Residence(Urban-1, Rural-2)
    
    "MV463A",  # Smokes cigarettes
    "MV463B",  # Smokes pipe
    "MV463C",  # Uses chewing tobacco
    "MV463D",  # Uses snuff
    "MV463X",  # Smokes other
    "MV463Z",  # Smokes nothing    
    
    "BMI",     # recovered BMI
    
    "SM500C1", # SBP2 (2nd measurement: Systolic)
    "SM500C2", # DBP2 (2nd measurement: Diastolic)
    "SM862A",  # SBP3 (3rd measurement: Systolic)
    "SM862B",  # DBP3 (3rd measurement: Diastolic)
]

In [107]:
# Check that every required variable exists

missing = [v for v in gm_vars if v not in gm_df.columns]
print(missing)

[]


In [108]:
# Extract only variables required for HyperSense

gm_extract = gm_df[gm_vars].copy()
print(gm_extract.shape)
gm_extract.head(5)

(4388, 19)


,MCASEID,MV001,MV002,MV003,MV005,MV012,MV106,MV025,MV463A,MV463B,MV463C,MV463D,MV463X,MV463Z,BMI,SM500C1,SM500C2,SM862A,SM862B
0,1 1 1,1.0,1.0,1.0,856663.0,50.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,1703.0,152.0,106.0,154.0,101.0
1,1 3 1,1.0,3.0,1.0,856663.0,27.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,2048.0,133.0,79.0,136.0,82.0
2,1 6 1,1.0,6.0,1.0,856663.0,24.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,2108.0,109.0,63.0,108.0,63.0
3,111 1,1.0,11.0,1.0,856663.0,40.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,1972.0,117.0,84.0,117.0,82.0
4,119 1,1.0,19.0,1.0,856663.0,43.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,2300.0,131.0,85.0,121.0,80.0


In [109]:
# Add gender identifier

gm_extract["gender"] = "Male"

In [110]:
# Calculate mean systolic and diastolic BP

gm_extract["mean_sbp"] = gm_extract[
    ["SM500C1", "SM862A"]
].mean(axis=1)

gm_extract["mean_dbp"] = gm_extract[
    ["SM500C2", "SM862B"]
].mean(axis=1)

In [111]:
# Hypertension defined from BP readings
# Initialize as missing
gm_extract["htn_status"] = pd.NA

# Only classify respondents with BP measurements
mask = (
    gm_extract["mean_sbp"].notna() &
    gm_extract["mean_dbp"].notna()
)

gm_extract.loc[mask, "htn_status"] = (
    (gm_extract.loc[mask, "mean_sbp"] >= 140) |
    (gm_extract.loc[mask, "mean_dbp"] >= 90)
).astype(int)

In [112]:
# Inspect class distribution

gm_extract["htn_status"].value_counts()

htn_status
0    3866
1     511
Name: count, dtype: int64

In [113]:
# Mean BP by HTN group

gm_extract.groupby("htn_status")[
    ["mean_sbp", "mean_dbp"]
].mean()

,mean_sbp,mean_dbp
htn_status,,
0,114.501811,71.701759
1,148.189824,94.622309


In [114]:
# Missing values in new variables

gm_extract[
    ["mean_sbp", "mean_dbp", "htn_status"]
].isna().sum()

mean_sbp      11
mean_dbp      11
htn_status    11
dtype: int64

In [115]:
gm_extract.head(5)

,MCASEID,MV001,MV002,MV003,MV005,MV012,MV106,MV025,MV463A,MV463B,...,MV463Z,BMI,SM500C1,SM500C2,SM862A,SM862B,gender,mean_sbp,mean_dbp,htn_status
0,1 1 1,1.0,1.0,1.0,856663.0,50.0,2.0,2.0,0.0,0.0,...,1.0,1703.0,152.0,106.0,154.0,101.0,Male,153.0,103.5,1
1,1 3 1,1.0,3.0,1.0,856663.0,27.0,2.0,2.0,0.0,0.0,...,1.0,2048.0,133.0,79.0,136.0,82.0,Male,134.5,80.5,0
2,1 6 1,1.0,6.0,1.0,856663.0,24.0,2.0,2.0,0.0,0.0,...,1.0,2108.0,109.0,63.0,108.0,63.0,Male,108.5,63.0,0
3,111 1,1.0,11.0,1.0,856663.0,40.0,1.0,2.0,0.0,0.0,...,0.0,1972.0,117.0,84.0,117.0,82.0,Male,117.0,83.0,0
4,119 1,1.0,19.0,1.0,856663.0,43.0,2.0,2.0,0.0,0.0,...,1.0,2300.0,131.0,85.0,121.0,80.0,Male,126.0,82.5,0


In [116]:
# Save Ghana Men extraction

gm_extract.to_csv(
    "../outputs/ghana_men_extract.csv",
    index=False
)

print("Ghana Men extraction saved successfully.")
print(gm_extract.shape)

Ghana Men extraction saved successfully.
(4388, 23)


### Step 3 — Ghana Women (IR)

In [117]:
# Load Ghana Women Recode (IR)

gw_df, meta = pyreadstat.read_sav(r"C:\Users\USER\Desktop\HyperSense\data\Ghana 2014\GHIR72SV\GHIR72FL.SAV")
print(gw_df.shape)

(9396, 4079)


In [118]:
# --------------------------------------------------
# Variables required for Ghana Women dataset
# --------------------------------------------------

gw_vars = [
    "CASEID", # Case Identification
    "V001",   # cluster
    "V002",   # household
    "V003",   # respondent line number
    "V005",   # Women's sample weight

    "V012",   # age
    "V106",   # education
    "V025",   # Residence(Urban-1, Rural-2)
    
    "V463A",  # Smokes cigarettes
    "V463B",  # Smokes pipe
    "V463C",  # Uses chewing tobacco
    "V463D",  # Uses snuff
    "V463X",  # Smokes other
    "V463Z",  # Smokes nothing    
    
    "V445",     # Measured BMI
    
    "S600CA", # SBP2 (2nd measurement: Systolic)
    "S600CB", # DBP2 (2nd measurement: Diastolic)
    "S1056A",  # SBP3 (3rd measurement: Systolic)
    "S1056B",  # DBP3 (3rd measurement: Diastolic)
]

In [119]:
# Check that all required variables exist

missing = [v for v in gw_vars if v not in gw_df.columns]
print(missing)

[]


In [120]:
# Extract only variables required for HyperSense

gw_extract = gw_df[gw_vars].copy()
print(gw_extract.shape)
gw_extract.head()

(9396, 19)


,CASEID,V001,V002,V003,V005,V012,V106,V025,V463A,V463B,V463C,V463D,V463X,V463Z,V445,S600CA,S600CB,S1056A,S1056B
0,1 1 2,1.0,1.0,2.0,858896.0,42.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,2026.0,115.0,66.0,118.0,63.0
1,1 3 2,1.0,3.0,2.0,858896.0,25.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,2360.0,121.0,90.0,117.0,91.0
2,1 5 2,1.0,5.0,2.0,858896.0,37.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,104.0,71.0,94.0,66.0
3,1 5 3,1.0,5.0,3.0,858896.0,15.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,127.0,85.0,124.0,85.0
4,1 6 2,1.0,6.0,2.0,858896.0,19.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,2620.0,88.0,69.0,107.0,67.0


In [121]:
# Add gender identifier

gw_extract["gender"] = "Female"

In [122]:
# Calculate mean systolic and diastolic BP

gw_extract["mean_sbp"] = gw_extract[
    ["S600CA", "S1056A"]
].mean(axis=1)

gw_extract["mean_dbp"] = gw_extract[
    ["S600CB", "S1056B"]
].mean(axis=1)

In [123]:
# Hypertension defined from BP readings
# Initialize as missing
gw_extract["htn_status"] = pd.NA

# Only classify respondents with BP measurements
mask = (
    gw_extract["mean_sbp"].notna() &
    gw_extract["mean_dbp"].notna()
)

gw_extract.loc[mask, "htn_status"] = (
    (gw_extract.loc[mask, "mean_sbp"] >= 140) |
    (gw_extract.loc[mask, "mean_dbp"] >= 90)
).astype(int)

In [124]:
# Check prevalence

gw_extract["htn_status"].value_counts()

htn_status
0    8525
1     844
Name: count, dtype: int64

In [125]:
# Check BP separation

gw_extract.groupby("htn_status")[
    ["mean_sbp", "mean_dbp"]
].mean()

,mean_sbp,mean_dbp
htn_status,,
0,106.943636,71.407097
1,143.843009,96.815758


In [126]:
# Missing values

gw_extract[
    ["mean_sbp", "mean_dbp", "htn_status"]
].isna().sum()

mean_sbp      27
mean_dbp      27
htn_status    27
dtype: int64

In [127]:
# Final shape before export

print(gw_extract.shape)
gw_extract.head()

(9396, 23)


,CASEID,V001,V002,V003,V005,V012,V106,V025,V463A,V463B,...,V463Z,V445,S600CA,S600CB,S1056A,S1056B,gender,mean_sbp,mean_dbp,htn_status
0,1 1 2,1.0,1.0,2.0,858896.0,42.0,1.0,2.0,0.0,0.0,...,1.0,2026.0,115.0,66.0,118.0,63.0,Female,116.5,64.5,0
1,1 3 2,1.0,3.0,2.0,858896.0,25.0,2.0,2.0,0.0,0.0,...,1.0,2360.0,121.0,90.0,117.0,91.0,Female,119.0,90.5,1
2,1 5 2,1.0,5.0,2.0,858896.0,37.0,1.0,2.0,0.0,0.0,...,1.0,NaN,104.0,71.0,94.0,66.0,Female,99.0,68.5,0
3,1 5 3,1.0,5.0,3.0,858896.0,15.0,1.0,2.0,0.0,0.0,...,1.0,NaN,127.0,85.0,124.0,85.0,Female,125.5,85.0,0
4,1 6 2,1.0,6.0,2.0,858896.0,19.0,2.0,2.0,0.0,0.0,...,1.0,2620.0,88.0,69.0,107.0,67.0,Female,97.5,68.0,0


In [128]:
# Save Ghana Women extraction

gw_extract.to_csv(
    "../outputs/ghana_women_extract.csv",
    index=False
)

print("Ghana Women extraction saved successfully.")
print(gw_extract.shape)

Ghana Women extraction saved successfully.
(9396, 23)


### Step 4 — Benin Men (MR)

In [129]:
# Load Benin Men Recode (MR)

bm_df, meta = pyreadstat.read_sav(
    r"C:\Users\USER\Desktop\HyperSense\data\Benin 2017-18\BJMR71SV\BJMR71FL.SAV"
)
print(bm_df.shape)

(7595, 621)


In [130]:
# --------------------------------------------------
# Variables required for Benin Men dataset
# --------------------------------------------------

bm_vars = [
    "MCASEID", # Case Identification
    "MV001",   # cluster
    "MV002",   # household
    "MV003",   # respondent line number
    "MV005",   # Men's sample weight

    "MV012",   # age
    "MV106",   # education
    "MV025",   # Residence(Urban-1, Rural-2)
    
    "MV463AA", # Frequency currently smokes tobacco
    "MV463AB", # Frequency currently uses smokeless tobacco  
        
    "SM936S",  # Mean SBP
    "SM936D",  # Mean DBP
]

In [131]:
# Check that all required variables exist

missing = [v for v in bm_vars if v not in bm_df.columns]
print(missing)

[]


In [132]:
# Extract required variables
bm_extract = bm_df[bm_vars].copy()

# BMI not collected in Benin Men DHS
bm_extract["BMI"] = pd.NA

# Gender
bm_extract["gender"] = "Male"

In [133]:
# Hypertension defined from BP readings
# Initialize as missing
bm_extract["htn_status"] = pd.NA

# Only classify respondents with BP measurements
mask = (
    bm_extract["SM936S"].notna() &
    bm_extract["SM936D"].notna()
)

bm_extract.loc[mask, "htn_status"] = (
    (bm_extract.loc[mask, "SM936S"] >= 140) |
    (bm_extract.loc[mask, "SM936D"] >= 90)
).astype(int)

In [134]:
# HTN prevalence

bm_extract["htn_status"].value_counts()

htn_status
0    3038
1     613
Name: count, dtype: int64

In [135]:
# Check for missing values

bm_extract[
    ["BMI", "SM936S", "SM936D", "htn_status"]
].isna().sum()

BMI           7595
SM936S        3944
SM936D        3944
htn_status    3944
dtype: int64

In [136]:
print(bm_extract.shape)
bm_extract.head()

(7595, 15)


,MCASEID,MV001,MV002,MV003,MV005,MV012,MV106,MV025,MV463AA,MV463AB,SM936S,SM936D,BMI,gender,htn_status
0,1 3 1,1.0,3.0,1.0,1130086.0,31.0,0.0,2.0,0.0,1.0,106.0,66.0,<NA>,Male,0
1,1 3 8,1.0,3.0,8.0,1130086.0,15.0,0.0,2.0,0.0,1.0,NaN,NaN,<NA>,Male,<NA>
2,1 15 3,1.0,15.0,3.0,1130086.0,45.0,0.0,2.0,0.0,0.0,124.0,89.0,<NA>,Male,0
3,1 15 9,1.0,15.0,9.0,1130086.0,42.0,0.0,2.0,0.0,0.0,115.0,80.0,<NA>,Male,0
4,1 15 11,1.0,15.0,11.0,1130086.0,15.0,1.0,2.0,0.0,0.0,NaN,NaN,<NA>,Male,<NA>


In [137]:
# Exploring suspected implausible BP values
bm_extract[["SM936S", "SM936D"]].describe()

,SM936S,SM936D
count,3651.000000,3651.000000
mean,125.149000,80.989318
std,56.649871,57.942566
min,71.000000,34.000000
25%,111.000000,70.000000
50%,120.000000,77.000000
75%,129.000000,83.000000
max,997.000000,997.000000


In [138]:
# Recode DHS special BP codes in Benin Men to missing

bm_extract["SM936S"] = bm_extract["SM936S"].replace(
    [994, 995, 996, 997, 998, 999],
    pd.NA
)

bm_extract["SM936D"] = bm_extract["SM936D"].replace(
    [994, 995, 996, 997, 998, 999],
    pd.NA
)

In [139]:
# Create HTN status only where BP exists

bm_extract["htn_status"] = pd.NA

mask = (
    bm_extract["SM936S"].notna() &
    bm_extract["SM936D"].notna()
)

bm_extract.loc[mask, "htn_status"] = (
    (bm_extract.loc[mask, "SM936S"] >= 140) |
    (bm_extract.loc[mask, "SM936D"] >= 90)
).astype(int)

In [140]:
# Check missingness

bm_extract[
    ["SM936S", "SM936D", "htn_status"]
].isna().sum()

SM936S        3958
SM936D        3958
htn_status    3958
dtype: int64

In [141]:
# Check BP separation

bm_extract.groupby("htn_status")[
    ["SM936S", "SM936D"]
].mean()

,SM936S,SM936D
htn_status,,
0,116.536208,74.129691
1,148.45409,94.370618


In [142]:
# Quick audit before export
bm_extract.isna().mean().sort_values(ascending=False) * 100

BMI           100.000000
SM936S         52.113232
SM936D         52.113232
htn_status     52.113232
MCASEID         0.000000
MV001           0.000000
MV002           0.000000
MV106           0.000000
MV012           0.000000
MV005           0.000000
MV003           0.000000
MV463AB         0.000000
MV463AA         0.000000
MV025           0.000000
gender          0.000000
dtype: float64

In [143]:
bm_extract.to_csv(
    "../outputs/benin_men_extract.csv",
    index=False
)

print("Benin Men extraction saved successfully.")
print(bm_extract.shape)

Benin Men extraction saved successfully.
(7595, 15)


##### NOTE:
- BMI was not collected in Benin Men (MR).
- Tobacco type variables are absent.
- Tobacco frequency variables (MV463AA, MV463AB) retained instead.
- Harmonized `tobacco_use` will be created later.

### Step 5 — Benin Women (IR)

In [144]:
# Load Benin Women Recode (IR)

bw_df, meta = pyreadstat.read_sav(
    r"C:\Users\USER\Desktop\HyperSense\data\Benin 2017-18\BJIR71SV\BJIR71FL.SAV"
)
print(bw_df.shape)

(15928, 5109)


In [145]:
# --------------------------------------------------
# Variables required for Benin Women dataset
# --------------------------------------------------

bw_vars = [
    "CASEID", # Case Identification
    "V001",   # cluster
    "V002",   # household
    "V003",   # respondent line number
    "V005",   # Women's sample weight

    "V012",   # age
    "V106",   # education
    "V025",   # Residence(Urban-1, Rural-2)
    
    "V463A",  # Smokes cigarettes
    "V463B",  # Smokes pipe
    "V463C",  # Uses chewing tobacco
    "V463D",  # Uses snuff
    "V463X",  # Smokes other
    "V463Z",  # Smokes nothing    
    
    "V463AA", # Frequency currently smokes tobacco
    "V463AB", # Frequency currently uses smokeless tobacco  
    
    "V445",     # Measured BMI
    
    "S1435S",  # Mean SBP
    "S1435D",  # Mean DBP
]

In [146]:
# Check that all required variables exist

missing = [v for v in bw_vars if v not in bw_df.columns]
print(missing)

[]


In [147]:
# Extract required variables

bw_extract = bw_df[bw_vars].copy()
print(bw_extract.shape)
bw_extract.head()

# Gender
bw_extract["gender"] = "Female"

(15928, 19)


In [148]:
# Create HTN status only where BP exists

bw_extract["htn_status"] = pd.NA

mask = (
    bw_extract["S1435S"].notna() &
    bw_extract["S1435D"].notna()
)

bw_extract.loc[mask, "htn_status"] = (
    (bw_extract.loc[mask, "S1435S"] >= 140) |
    (bw_extract.loc[mask, "S1435D"] >= 90)
).astype(int)

In [149]:
# HTN Prevalence
print(bw_extract['htn_status'].value_counts())

# Check missing variables
print(bw_extract[["S1435S", "S1435D", "htn_status"]].isna().sum())

htn_status
0    2668
1     511
Name: count, dtype: int64
S1435S        12749
S1435D        12749
htn_status    12749
dtype: int64


In [150]:
# Check BP separation for this too

bw_extract.groupby("htn_status")[["S1435S", "S1435D"]].mean()

,S1435S,S1435D
htn_status,,
0,109.059595,71.834708
1,340.604697,295.499022


In [151]:
# Explore reasons for such high averages for those with HTN
bw_extract[['S1435S', 'S1435D']].describe()

,S1435S,S1435D
count,3179.000000,3179.000000
mean,146.278704,107.787040
std,166.508083,171.903707
min,78.000000,45.000000
25%,103.000000,67.000000
50%,111.000000,74.000000
75%,122.000000,82.000000
max,997.000000,997.000000


In [152]:
# Recode DHS special BP codes to missing

bw_extract["S1435S"] = bw_extract["S1435S"].replace(
    [994, 995, 996, 997, 998, 999],
    pd.NA
)

bw_extract["S1435D"] = bw_extract["S1435D"].replace(
    [994, 995, 996, 997, 998, 999],
    pd.NA
)

In [153]:
# Recreate HTN status after cleaning BP variables

bw_extract["htn_status"] = pd.NA

mask = (
    bw_extract["S1435S"].notna() &
    bw_extract["S1435D"].notna()
)

bw_extract.loc[mask, "htn_status"] = (
    (bw_extract.loc[mask, "S1435S"] >= 140) |
    (bw_extract.loc[mask, "S1435D"] >= 90)
).astype(int)

In [ ]:
# Check BP separation again (sanity check!)

bw_extract.groupby("htn_status")[["S1435S", "S1435D"]].mean()

,S1435S,S1435D
htn_status,,
0,109.059595,71.834708
1,147.840506,94.096203


Benin BP variables contained DHS special codes (994–999) that were recoded to missing prior to HTN classification.

In [155]:
# Final audit

bw_extract.isna().mean().sort_values(ascending=False) * 100

S1435S        80.769714
htn_status    80.769714
S1435D        80.757157
V445          49.384731
CASEID         0.000000
V005           0.000000
V003           0.000000
V002           0.000000
V001           0.000000
V012           0.000000
V106           0.000000
V025           0.000000
V463A          0.000000
V463X          0.000000
V463D          0.000000
V463C          0.000000
V463B          0.000000
V463AB         0.000000
V463AA         0.000000
V463Z          0.000000
gender         0.000000
dtype: float64

In [157]:
# Save final individual output before harmonization!

bw_extract.to_csv(
    "../outputs/benin_women_extract.csv",
    index=False
)

print("Benin Women extraction saved successfully.")
print(bw_extract.shape)

Benin Women extraction saved successfully.
(15928, 21)
